# Result Analysis

This notebook compares FLEXIMOD scenario outputs from existing `data/output/` folders. It does not rerun simulations. The focus is the economic result by scenario: gas usage cost, electricity procurement from different markets, additional electricity charges, market revenues, and net operating cost.

## 1. Imports And Paths

This cell prepares the notebook environment, locates the project root, and sets a readable seaborn/matplotlib style for report-ready plots.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Find the FLEXIMOD repository root from the current notebook location."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "data" / "output"
FIGURE_DIR = PROJECT_ROOT / "notebooks" / "figures"

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 220,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")

## 2. Select Scenarios

By default, the notebook analyses the standard scenario outputs in a fixed order. To analyse only selected cases, replace `SELECTED_SCENARIOS = None` with a list of scenario names.

In [ ]:
STRATEGY_NAME = "hybrid_etes_gas"

SCENARIO_ORDER = [
    "hybrid_ETES_DA",
    "hybrid_ETES_DA_ID_buy",
    "hybrid_ETES_DA_ID_buy_sell",
    "hybrid_ETES_DA_ID_aFRR_energy",
    "hybrid_ETES_DA_ID_aFRR_energy_capacity",
]

# Set to a list such as ["hybrid_ETES_DA", "hybrid_ETES_DA_ID_buy"]
# if you want to analyse only selected scenarios.
SELECTED_SCENARIOS: list[str] | None = None

# Keep figures display-only by default. Set True if you want PNG exports in notebooks/figures/.
SAVE_FIGURES = False


def output_folder_for(scenario: str) -> Path:
    return OUTPUT_ROOT / f"{scenario}_{STRATEGY_NAME}"


def discover_scenario_paths(selected: list[str] | None = None) -> dict[str, Path]:
    available = {
        folder.name.removesuffix(f"_{STRATEGY_NAME}"): folder
        for folder in OUTPUT_ROOT.glob(f"*_{STRATEGY_NAME}")
        if folder.is_dir()
    }
    scenarios = selected or [name for name in SCENARIO_ORDER if name in available]
    missing = [name for name in scenarios if name not in available]
    if missing:
        warnings.warn(
            "Missing output folder(s): "
            + ", ".join(str(output_folder_for(name)) for name in missing),
            stacklevel=2,
        )
    return {name: available[name] for name in scenarios if name in available}


scenario_paths = discover_scenario_paths(SELECTED_SCENARIOS)
if not scenario_paths:
    raise RuntimeError("No scenario output folders were found for analysis.")

scenario_overview = pd.DataFrame(
    [
        {
            "scenario": name,
            "output_folder": str(path.relative_to(PROJECT_ROOT)),
            "has_summary": (path / "summary_indicators.csv").exists(),
            "has_dispatch": (path / "dispatch_results.csv").exists(),
            "has_market_ledger": (path / "market_ledger.csv").exists(),
            "has_afrr_capacity_blocks": (path / "afrr_capacity_block_summary.csv").exists(),
        }
        for name, path in scenario_paths.items()
    ]
)
display(scenario_overview)

## 3. Load Output Tables

This cell loads the already exported result files for each scenario. `summary_indicators.csv` provides scenario totals, while `dispatch_results.csv` is used to reconstruct market-specific cost components such as day-ahead market energy cost.

In [ ]:
def read_optional_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        warnings.warn(f"Missing result table: {path}", stacklevel=2)
        return pd.DataFrame()
    return pd.read_csv(path)


def load_scenario_outputs(paths: dict[str, Path]) -> dict[str, dict[str, pd.DataFrame]]:
    outputs: dict[str, dict[str, pd.DataFrame]] = {}
    for scenario, folder in paths.items():
        outputs[scenario] = {
            "summary": read_optional_csv(folder / "summary_indicators.csv"),
            "dispatch": read_optional_csv(folder / "dispatch_results.csv"),
            "market_ledger": read_optional_csv(folder / "market_ledger.csv"),
            "storage_cost_ledger": read_optional_csv(folder / "storage_cost_ledger.csv"),
        }
    return outputs


scenario_outputs = load_scenario_outputs(scenario_paths)
loaded_overview = pd.DataFrame(
    [
        {
            "scenario": scenario,
            "summary_rows": len(tables["summary"]),
            "dispatch_rows": len(tables["dispatch"]),
            "market_ledger_rows": len(tables["market_ledger"]),
            "storage_cost_rows": len(tables["storage_cost_ledger"]),
        }
        for scenario, tables in scenario_outputs.items()
    ]
)
display(loaded_overview)

## 4. Build Cost And Revenue Table

This cell creates the economic comparison table. Costs are kept positive, revenues are kept positive in the table, and the recomputed net cost subtracts revenues. The plotting cells show revenues below zero.

In [ ]:
def first_row_value(summary: pd.DataFrame, column: str, default: float = 0.0) -> float:
    if summary.empty or column not in summary.columns:
        return default
    value = pd.to_numeric(summary[column], errors="coerce").iloc[0]
    return float(value) if pd.notna(value) else default


def sum_column(df: pd.DataFrame, column: str) -> float:
    if df.empty or column not in df.columns:
        return 0.0
    return float(pd.to_numeric(df[column], errors="coerce").fillna(0.0).sum())


def nonzero_or_fallback(value: float, fallback: float) -> float:
    return value if abs(value) > 1e-9 else fallback


def scenario_cost_record(scenario: str, tables: dict[str, pd.DataFrame]) -> dict[str, float | str]:
    summary = tables["summary"]
    dispatch = tables["dispatch"]

    gas_cost = nonzero_or_fallback(
        first_row_value(summary, "total_gas_cost_EUR"),
        sum_column(dispatch, "gas_cost_EUR"),
    )
    da_cost = sum_column(dispatch, "DA_electricity_cost_EUR")
    idc_buy_cost = nonzero_or_fallback(
        first_row_value(summary, "IDC_buy_cost_EUR"),
        sum_column(dispatch, "IDC_buy_cost_EUR"),
    )
    idc_sell_revenue = nonzero_or_fallback(
        first_row_value(summary, "IDC_sell_revenue_EUR"),
        sum_column(dispatch, "IDC_sell_revenue_EUR"),
    )
    afrr_energy_cost = nonzero_or_fallback(
        first_row_value(summary, "afrr_energy_cost_EUR"),
        sum_column(dispatch, "afrr_energy_cost_EUR"),
    )
    additional_charges = nonzero_or_fallback(
        first_row_value(summary, "total_additional_electricity_charges_cost_EUR"),
        sum_column(dispatch, "additional_electricity_charges_cost_EUR"),
    )
    capacity_revenue = nonzero_or_fallback(
        first_row_value(summary, "total_afrr_capacity_revenue_EUR"),
        sum_column(dispatch, "afrr_capacity_revenue_EUR"),
    )

    gross_cost = gas_cost + da_cost + idc_buy_cost + afrr_energy_cost + additional_charges
    net_cost = gross_cost - idc_sell_revenue - capacity_revenue
    model_net_cost = first_row_value(
        summary,
        "net_operating_cost_EUR",
        first_row_value(summary, "total_net_operating_cost_EUR", np.nan),
    )

    return {
        "scenario": scenario,
        "plant_name": str(summary["plant_name"].iloc[0])
        if not summary.empty and "plant_name" in summary.columns
        else "",
        "gas_usage_cost_EUR": gas_cost,
        "DA_market_energy_cost_EUR": da_cost,
        "IDC_buy_cost_EUR": idc_buy_cost,
        "afrr_energy_cost_EUR": afrr_energy_cost,
        "additional_electricity_charges_EUR": additional_charges,
        "IDC_sell_revenue_EUR": idc_sell_revenue,
        "afrr_capacity_revenue_EUR": capacity_revenue,
        "gross_cost_recomputed_EUR": gross_cost,
        "net_cost_recomputed_EUR": net_cost,
        "model_net_cost_EUR": model_net_cost,
        "net_cost_difference_EUR": net_cost - model_net_cost
        if pd.notna(model_net_cost)
        else np.nan,
        "total_heat_demand_MWh": first_row_value(summary, "total_heat_demand_MWh", np.nan),
        "total_DA_electricity_MWh": first_row_value(summary, "total_DA_electricity_MWh", np.nan),
        "total_IDC_buy_MWh": first_row_value(summary, "total_IDC_buy_MWh", np.nan),
        "total_IDC_sell_MWh": first_row_value(summary, "total_IDC_sell_MWh", np.nan),
        "total_afrr_energy_activated_MWh": first_row_value(
            summary, "total_afrr_energy_activated_MWh", np.nan
        ),
        "total_afrr_capacity_reserved_MW_h": first_row_value(
            summary, "total_afrr_capacity_reserved_MW_h", np.nan
        ),
        "average_cost_of_heat_EUR_per_MWh": first_row_value(
            summary, "average_cost_of_heat_EUR_per_MWh", np.nan
        ),
    }


analysis_df = pd.DataFrame(
    [scenario_cost_record(scenario, tables) for scenario, tables in scenario_outputs.items()]
)
analysis_df["net_DA_after_IDC_sell_MWh"] = (
    pd.to_numeric(analysis_df["total_DA_electricity_MWh"], errors="coerce").fillna(0.0)
    - pd.to_numeric(analysis_df["total_IDC_sell_MWh"], errors="coerce").fillna(0.0)
).clip(lower=0.0)

analysis_df["scenario"] = pd.Categorical(
    analysis_df["scenario"], categories=list(scenario_paths), ordered=True
)
analysis_df = analysis_df.sort_values("scenario").reset_index(drop=True)

large_mismatch = analysis_df["net_cost_difference_EUR"].abs() > 1.0
if large_mismatch.any():
    warnings.warn(
        "Recomputed net cost differs from model net cost by more than 1 EUR for: "
        + ", ".join(analysis_df.loc[large_mismatch, "scenario"].astype(str)),
        stacklevel=2,
    )

cost_columns = [
    "gas_usage_cost_EUR",
    "DA_market_energy_cost_EUR",
    "IDC_buy_cost_EUR",
    "afrr_energy_cost_EUR",
    "additional_electricity_charges_EUR",
    "IDC_sell_revenue_EUR",
    "afrr_capacity_revenue_EUR",
    "gross_cost_recomputed_EUR",
    "net_cost_recomputed_EUR",
    "model_net_cost_EUR",
    "net_cost_difference_EUR",
]
display(
    analysis_df[["scenario", *cost_columns]].style.format(
        {column: "{:,.0f}" for column in cost_columns}
    )
)

## 5. Cost And Revenue Activity Plot Per Scenario

Each scenario is shown as grouped activity bars rather than one stacked cost bar. Costs are positive bars. The revenue bar stacks IDC sell revenue and aFRR capacity revenue so revenue remains clearly separate from procurement costs.

In [ ]:
ACTIVITY_COST_BARS = [
    ("gas_usage_cost_EUR", "Gas cost"),
    ("DA_market_energy_cost_EUR", "DA market cost\n(excl. charges)"),
    ("IDC_buy_cost_EUR", "IDC buy cost"),
    ("afrr_energy_cost_EUR", "aFRR energy settlement"),
    ("additional_electricity_charges_EUR", "Electricity\ncharges"),
]
REVENUE_STACK = [
    ("IDC_sell_revenue_EUR", "IDC sell revenue"),
    ("afrr_capacity_revenue_EUR", "aFRR capacity revenue"),
]

COLORS = {
    "Gas cost": "#C44E52",
    "DA market cost\n(excl. charges)": "#4C72B0",
    "IDC buy cost": "#55A868",
    "aFRR energy settlement": "#8172B2",
    "Electricity\ncharges": "#DD8452",
    "IDC sell revenue": "#64B5CD",
    "aFRR capacity revenue": "#937860",
    "Net operating cost": "#111111",
    "aFRR capacity": "#222222",
}


def scenario_label(scenario: str) -> str:
    labels = {
        "hybrid_ETES_DA": "DA",
        "hybrid_ETES_DA_ID_buy": "DA + IDC buy-only",
        "hybrid_ETES_DA_ID_buy_sell": "DA + IDC buy/sell",
        "hybrid_ETES_DA_ID_aFRR_energy": "DA + IDC + aFRR energy",
        "hybrid_ETES_DA_ID_aFRR_energy_capacity": "DA + IDC + aFRR energy + capacity",
    }
    return labels.get(scenario, scenario)


def _deduplicate_legend(ax: plt.Axes, loc: str = "upper left") -> None:
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles, strict=False))
    ax.legend(unique.values(), unique.keys(), loc=loc, bbox_to_anchor=(1.02, 1.0), frameon=True)


def _save_figure(fig: plt.Figure, filename: str) -> None:
    if not SAVE_FIGURES:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / filename, bbox_inches="tight")


COST_SCALE_EUR = 1_000.0
COST_UNIT = "kEUR"


def _format_cost(value: float) -> str:
    if abs(value) >= 100:
        return f"{value:,.0f}"
    return f"{value:,.1f}"


def _use_plain_cost_ticks(ax: plt.Axes) -> None:
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))


def _annotate_bar(ax: plt.Axes, x: float, y: float, text: str, *, positive: bool = True) -> None:
    va = "bottom" if positive else "top"
    offset = 0.02 if positive else -0.02
    ax.text(x, y + offset, text, ha="center", va=va, fontsize=9, color="#111111")


def plot_activity_cost_revenue(row: pd.Series, scale: float = COST_SCALE_EUR) -> plt.Figure:
    scenario = str(row["scenario"])
    fig, ax = plt.subplots(figsize=(12, 6))

    labels = [label for _, label in ACTIVITY_COST_BARS] + ["Revenue"]
    x = np.arange(len(labels))
    width = 0.62

    for position, (column, label) in enumerate(ACTIVITY_COST_BARS):
        value = float(row[column]) / scale
        ax.bar(position, value, width, color=COLORS[label], label=label.replace("\n", " "))
        if abs(value) > 1e-9:
            _annotate_bar(ax, position, value, _format_cost(value), positive=value >= 0)

    revenue_position = len(labels) - 1
    revenue_bottom = 0.0
    total_revenue = 0.0
    for column, label in REVENUE_STACK:
        value = float(row[column]) / scale
        if abs(value) < 1e-9:
            continue
        ax.bar(
            revenue_position, value, width, bottom=revenue_bottom, color=COLORS[label], label=label
        )
        revenue_bottom += value
        total_revenue += value
    if abs(total_revenue) > 1e-9:
        _annotate_bar(
            ax, revenue_position, total_revenue, _format_cost(total_revenue), positive=True
        )

    net_cost = float(row["net_cost_recomputed_EUR"]) / scale
    ax.axhline(
        net_cost,
        color=COLORS["Net operating cost"],
        linestyle="--",
        linewidth=1.8,
        label="Net operating cost",
    )
    ax.text(
        0.98,
        net_cost,
        f"Net: {_format_cost(net_cost)}",
        transform=ax.get_yaxis_transform(),
        va="bottom",
        ha="right",
        fontsize=10,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 2},
        clip_on=True,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel(f"Cost / revenue [{COST_UNIT}]")
    _use_plain_cost_ticks(ax)
    ax.set_title(f"Cost and revenue activities: {scenario_label(scenario)}")
    ax.text(
        0.0,
        1.02,
        "DA and IDC bars are market energy costs. aFRR energy is a settlement: "
        "positive values are costs, negative values are credits. "
        "Electricity charges are shown separately.",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9,
        color="#444444",
    )
    _deduplicate_legend(ax)
    fig.tight_layout()
    _save_figure(fig, f"activity_cost_revenue_{scenario}.png")
    return fig


for _, scenario_row in analysis_df.iterrows():
    fig = plot_activity_cost_revenue(scenario_row)
    plt.show()

## 6. Trade Activity Volumes Across Scenarios

This figure compares the physical market activity by scenario. Energy procurement and reductions are shown in `MWh_el` on the left axis. aFRR capacity reservation is shown separately on the right axis in `MW?h` because it is not an energy volume.

In [ ]:
VOLUME_BARS = [
    ("total_DA_electricity_MWh", "Gross DA procured"),
    ("total_IDC_sell_MWh", "DA sold/reduced in IDC"),
    ("net_DA_after_IDC_sell_MWh", "Net DA retained"),
    ("total_IDC_buy_MWh", "IDC procured"),
    ("total_afrr_energy_activated_MWh", "aFRR energy activated"),
]
VOLUME_COLORS = {
    "Gross DA procured": "#4C72B0",
    "DA sold/reduced in IDC": "#64B5CD",
    "Net DA retained": "#2F4B7C",
    "IDC procured": "#55A868",
    "aFRR energy activated": "#8172B2",
    "aFRR capacity reserved": "#222222",
}


def plot_trade_activity_volumes(df: pd.DataFrame) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(len(df))
    group_width = 0.78
    bar_width = group_width / len(VOLUME_BARS)
    offsets = np.linspace(
        -group_width / 2 + bar_width / 2, group_width / 2 - bar_width / 2, len(VOLUME_BARS)
    )

    for offset, (column, label) in zip(offsets, VOLUME_BARS, strict=False):
        values = pd.to_numeric(df[column], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        alpha = 0.55 if label == "Gross DA procured" else 0.95
        edgecolor = "#1F2A44" if label == "Gross DA procured" else "none"
        ax.bar(
            x + offset,
            values,
            bar_width,
            color=VOLUME_COLORS[label],
            alpha=alpha,
            edgecolor=edgecolor,
            linewidth=1.0,
            label=label,
        )

    ax2 = ax.twinx()
    capacity = (
        pd.to_numeric(df["total_afrr_capacity_reserved_MW_h"], errors="coerce")
        .fillna(0.0)
        .to_numpy(dtype=float)
    )
    ax2.bar(
        x + group_width / 2 + 0.08,
        capacity,
        bar_width * 0.75,
        facecolor="none",
        edgecolor=VOLUME_COLORS["aFRR capacity reserved"],
        hatch="///",
        linewidth=1.5,
        label="aFRR capacity reserved",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [scenario_label(str(name)) for name in df["scenario"]], rotation=20, ha="right"
    )
    ax.set_ylabel("Energy volume [MWh_el]")
    ax2.set_ylabel("aFRR capacity reservation [MW?h]")
    ax.set_title("Trade activity volumes by scenario")
    ax.text(
        0.0,
        1.02,
        "Gross DA is the fixed DA market position. Net DA retained subtracts "
        "the DA volume sold/reduced in IDC.",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9,
        color="#444444",
    )

    handles_1, labels_1 = ax.get_legend_handles_labels()
    handles_2, labels_2 = ax2.get_legend_handles_labels()
    ax.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="upper left",
        bbox_to_anchor=(1.08, 1.0),
        frameon=True,
    )
    fig.tight_layout()
    _save_figure(fig, "trade_activity_volumes_by_scenario.png")
    return fig


volume_fig = plot_trade_activity_volumes(analysis_df)
plt.show()

## 7. Active Week Trade Profiles Across Scenarios

This cell selects one comparable week across all scenarios and shows the within-week market profile for each use case. By default, the week is chosen to include as much relevant activity as possible across the scenarios: IDC buy/sell, aFRR energy activation, and aFRR capacity reservation. Set `PROFILE_WEEK_START = "YYYY-MM-DD"` to force a specific week.


In [ ]:
PROFILE_WEEK_START = None  # Example: "2025-01-13". Keep None for automatic active-week selection.
PROFILE_TIMEZONE = "Europe/Berlin"
PROFILE_DAYS = 7

TRADE_PROFILE_COLORS = {
    "DA position": "#4C72B0",
    "IDC buy": "#55A868",
    "IDC sell/reduction": "#C44E52",
    "aFRR activated": "#8172B2",
    "aFRR capacity reserved": "#222222",
    "Scheduled DA+IDC": "#222222",
    "Actual electricity": "#000000",
}


def _datetime_indexed_market_frame(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    market = tables["market_ledger"].copy()
    if market.empty:
        market = tables["dispatch"].rename(
            columns={
                "DA_position_MWh": "day_ahead_position_MWh_el",
                "IDC_buy_MWh": "intraday_buy_MWh_el",
                "IDC_sell_MWh": "intraday_sell_MWh_el",
                "final_planned_electricity_MWh": "scheduled_electricity_procurement_MWh_el",
                "afrr_energy_activated_MWh": "afrr_energy_activated_MWh_el",
                "actual_electricity_consumption_MWh": "actual_electricity_consumption_MWh_el",
                "afrr_capacity_reserved_MW": "afrr_capacity_reserved_MW",
            }
        )
    if market.empty or "datetime" not in market.columns:
        return pd.DataFrame()
    market["datetime"] = pd.to_datetime(
        market["datetime"], errors="coerce", utc=True
    ).dt.tz_convert(PROFILE_TIMEZONE)
    market = market.dropna(subset=["datetime"]).set_index("datetime").sort_index()
    return market


def _series_or_zero(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(0.0, index=frame.index)
    return pd.to_numeric(frame[column], errors="coerce").fillna(0.0)


market_profiles = {
    scenario: _datetime_indexed_market_frame(tables)
    for scenario, tables in scenario_outputs.items()
}


def _profile_activity(frame: pd.DataFrame) -> pd.Series:
    if frame.empty:
        return pd.Series(dtype=float)
    capacity_mwh_equivalent = _series_or_zero(frame, "afrr_capacity_reserved_MWh")
    if capacity_mwh_equivalent.abs().sum() <= 1e-12:
        capacity_mwh_equivalent = _series_or_zero(
            frame, "afrr_capacity_reserved_MW"
        ) * _profile_timestep_hours(frame.index)
    activity = (
        _series_or_zero(frame, "intraday_buy_MWh_el").abs()
        + _series_or_zero(frame, "intraday_sell_MWh_el").abs()
        + _series_or_zero(frame, "afrr_energy_activated_MWh_el").abs()
        + capacity_mwh_equivalent.abs()
    )
    return activity.groupby(frame.index.floor("D")).sum()


def _profile_timestep_hours(index: pd.DatetimeIndex) -> float:
    if len(index) < 2:
        return 0.25
    diffs = pd.Series(index).diff().dropna()
    if diffs.empty:
        return 0.25
    return float(diffs.median() / pd.Timedelta(hours=1))


def select_profile_week(
    profiles: dict[str, pd.DataFrame],
    selected_week_start: str | None = None,
    days: int = PROFILE_DAYS,
) -> pd.Timestamp:
    available_days = [set(frame.index.floor("D")) for frame in profiles.values() if not frame.empty]
    if not available_days:
        raise ValueError("No market profiles are available for weekly profile plotting.")
    common_days = sorted(set.intersection(*available_days))
    if not common_days:
        raise ValueError("No common datetime day exists across all selected scenarios.")

    if selected_week_start is not None:
        selected = pd.Timestamp(selected_week_start).tz_localize(PROFILE_TIMEZONE)
        needed = {selected + pd.Timedelta(days=i) for i in range(days)}
        if not needed.issubset(set(common_days)):
            raise ValueError(
                f"Selected profile week {selected_week_start} is not available in all scenarios."
            )
        return selected

    daily_scores = pd.Series(0.0, index=pd.DatetimeIndex(common_days))
    for frame in profiles.values():
        daily = _profile_activity(frame)
        daily_scores = daily_scores.add(
            daily.reindex(daily_scores.index).fillna(0.0), fill_value=0.0
        )

    best_start = daily_scores.index[0]
    best_score = -np.inf
    common_set = set(daily_scores.index)
    for candidate in daily_scores.index:
        window_days = [candidate + pd.Timedelta(days=i) for i in range(days)]
        if not set(window_days).issubset(common_set):
            continue
        score = float(daily_scores.reindex(window_days).fillna(0.0).sum())
        if score > best_score:
            best_score = score
            best_start = candidate
    return pd.Timestamp(best_start)


def _window_slice(
    frame: pd.DataFrame, start: pd.Timestamp, days: int = PROFILE_DAYS
) -> pd.DataFrame:
    window_start = pd.Timestamp(start)
    if window_start.tzinfo is None:
        window_start = window_start.tz_localize(PROFILE_TIMEZONE)
    window_end = window_start + pd.Timedelta(days=days)
    return frame.loc[(frame.index >= window_start) & (frame.index < window_end)]


def _bar_width_days(index: pd.DatetimeIndex) -> float:
    if len(index) < 2:
        return 0.008
    diffs = pd.Series(index).diff().dropna()
    if diffs.empty:
        return 0.008
    return float(diffs.median() / pd.Timedelta(days=1)) * 0.85


def plot_active_week_trade_profiles(
    profiles: dict[str, pd.DataFrame],
    week_start: pd.Timestamp,
    days: int = PROFILE_DAYS,
) -> plt.Figure:
    scenario_names = list(profiles)
    fig, axes = plt.subplots(
        len(scenario_names),
        1,
        figsize=(17, 2.7 * len(scenario_names)),
        sharex=True,
        sharey=True,
    )
    if len(scenario_names) == 1:
        axes = [axes]

    y_min = 0.0
    y_max = 0.0
    prepared: dict[str, pd.DataFrame] = {}
    for scenario, frame in profiles.items():
        week_frame = _window_slice(frame, week_start, days)
        prepared[scenario] = week_frame
        if week_frame.empty:
            continue
        da = _series_or_zero(week_frame, "day_ahead_position_MWh_el")
        buy = _series_or_zero(week_frame, "intraday_buy_MWh_el")
        sell = _series_or_zero(week_frame, "intraday_sell_MWh_el")
        scheduled = _series_or_zero(week_frame, "scheduled_electricity_procurement_MWh_el")
        afrr = _series_or_zero(week_frame, "afrr_energy_activated_MWh_el")
        actual = _series_or_zero(week_frame, "actual_electricity_consumption_MWh_el")
        y_min = min(y_min, float((-sell).min()))
        y_max = max(y_max, float(pd.concat([da + buy, scheduled + afrr, actual]).max()))

    for ax, scenario in zip(axes, scenario_names, strict=False):
        week_frame = prepared[scenario]
        if week_frame.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(scenario_label(scenario), loc="left")
            continue

        times = week_frame.index
        width = _bar_width_days(times)
        da = _series_or_zero(week_frame, "day_ahead_position_MWh_el")
        buy = _series_or_zero(week_frame, "intraday_buy_MWh_el")
        sell = _series_or_zero(week_frame, "intraday_sell_MWh_el")
        scheduled = _series_or_zero(week_frame, "scheduled_electricity_procurement_MWh_el")
        afrr = _series_or_zero(week_frame, "afrr_energy_activated_MWh_el")
        actual = _series_or_zero(week_frame, "actual_electricity_consumption_MWh_el")
        capacity_mwh = _series_or_zero(week_frame, "afrr_capacity_reserved_MWh")

        ax.bar(
            times,
            da,
            width=width,
            color=TRADE_PROFILE_COLORS["DA position"],
            alpha=0.28,
            label="DA position",
        )
        ax.bar(
            times,
            buy,
            width=width,
            bottom=da,
            color=TRADE_PROFILE_COLORS["IDC buy"],
            alpha=0.72,
            label="IDC buy",
        )
        ax.bar(
            times,
            -sell,
            width=width,
            color=TRADE_PROFILE_COLORS["IDC sell/reduction"],
            alpha=0.72,
            label="IDC sell/reduction",
        )
        ax.bar(
            times,
            afrr,
            width=width,
            bottom=scheduled,
            color=TRADE_PROFILE_COLORS["aFRR activated"],
            alpha=0.72,
            label="aFRR activated",
        )
        if capacity_mwh.abs().sum() > 1e-12:
            ax.bar(
                times,
                capacity_mwh,
                width=width,
                bottom=actual,
                facecolor="none",
                edgecolor=TRADE_PROFILE_COLORS["aFRR capacity reserved"],
                hatch="///",
                linewidth=0.8,
                label="aFRR capacity reserved",
            )
        ax.step(
            times,
            scheduled,
            where="post",
            color=TRADE_PROFILE_COLORS["Scheduled DA+IDC"],
            linewidth=1.2,
            label="Scheduled DA+IDC",
        )
        ax.step(
            times,
            actual,
            where="post",
            color=TRADE_PROFILE_COLORS["Actual electricity"],
            linewidth=1.8,
            linestyle="--",
            label="Actual electricity",
        )
        ax.axhline(0.0, color="#333333", linewidth=0.8)
        ax.set_title(scenario_label(scenario), loc="left")
        ax.set_ylabel("MWh_el")
        ax.grid(True, alpha=0.25)

    if y_max > 0:
        margin = 0.08 * (y_max - y_min if y_max > y_min else y_max)
        for ax in axes:
            ax.set_ylim(y_min - margin, y_max + margin)
    axes[-1].set_xlabel("Time")
    axes[-1].xaxis.set_major_locator(mdates.DayLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes[1:]:
        axis_handles, axis_labels = ax.get_legend_handles_labels()
        handles.extend(axis_handles)
        labels.extend(axis_labels)
    unique = dict(zip(labels, handles, strict=False))
    fig.legend(
        unique.values(),
        unique.keys(),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=True,
    )
    window_end = week_start + pd.Timedelta(days=days)
    fig.suptitle(
        "Active week trade profiles: "
        f"{week_start.date()} to {(window_end - pd.Timedelta(days=1)).date()}",
        y=1.04,
    )
    fig.tight_layout()
    _save_figure(fig, f"active_week_trade_profiles_{week_start.date()}.png")
    return fig


profile_week_start = select_profile_week(market_profiles, PROFILE_WEEK_START)
print(
    "Selected profile week: "
    f"{profile_week_start.date()} to "
    f"{(profile_week_start + pd.Timedelta(days=PROFILE_DAYS - 1)).date()}"
)
active_week_fig = plot_active_week_trade_profiles(market_profiles, profile_week_start)
plt.show()

## 8. Net Operating Cost Across Scenarios

This final comparison shows one net operating cost bar per scenario. The net value includes gas cost, market energy costs, additional electricity charges, and subtracts IDC sell revenue and aFRR capacity revenue.

In [ ]:
def plot_net_cost_comparison(df: pd.DataFrame, scale: float = COST_SCALE_EUR) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    values = (
        pd.to_numeric(df["net_cost_recomputed_EUR"], errors="coerce").to_numpy(dtype=float) / scale
    )
    colors = sns.color_palette("deep", n_colors=len(df))
    ax.bar(x, values, color=colors, width=0.62)
    for xpos, value in zip(x, values, strict=False):
        _annotate_bar(ax, xpos, value, _format_cost(value), positive=value >= 0)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [scenario_label(str(name)) for name in df["scenario"]], rotation=20, ha="right"
    )
    ax.set_ylabel(f"Net operating cost [{COST_UNIT}]")
    _use_plain_cost_ticks(ax)
    ax.set_title("Net operating cost by scenario")
    ax.axhline(0, color="#333333", linewidth=1.0)
    fig.tight_layout()
    _save_figure(fig, "net_operating_cost_by_scenario.png")
    return fig


net_cost_fig = plot_net_cost_comparison(analysis_df)
plt.show()

## 9. Figure Export Status

The notebook displays figures by default. If `SAVE_FIGURES = True` in the scenario-selection cell, the figures are also written to `notebooks/figures/`.

In [ ]:
# if SAVE_FIGURES:
#     print(f"Figures saved to: {FIGURE_DIR}")
# else:
#     print("Figures were displayed only. Set SAVE_FIGURES = True to export PNG files.")